# 11 - Concern 2: confidence intervals and significance tests for every headline claim

**Reviewer concern.** The headline margins (fractions of a point) come from three splits, with no confidence
intervals and no significance tests, and may be within noise.

**What this notebook does (CPU, no search is repeated).**
1. Rebuilds every stored model from the configuration saved in the result files, on its original split, and
   checks that it reproduces the stored confusion counts **exactly** (`MATCH`). Only matching runs are analysed.
2. Gives a 95% stratified-bootstrap interval for each headline metric (F1, accuracy, ROC area, PR area, MCC).
3. Tests paired comparisons on identical test profiles: projection vs STE, HGS vs random search
   (DeLong for ROC areas, exact McNemar for errors, paired bootstrap for F1), and combines seeds with the
   Nadeau-Bengio corrected resampled t-test.
4. Compares each result with the published figure, both treating the published value as fixed and, where the
   published test size is known, accounting for its own sampling error (two-sample test).
5. Reports the smallest difference each comparison could detect (80% power), which tells the reader which
   claims the data can support at all.

**Runtime:** about 10-20 minutes on a CPU. **Outputs:** `stats_report.txt`, `stats_claims.csv`.

## Shared setup (identical in every job notebook)

Run these cells in order. The embeddings are loaded from the cache written by the first run, so a **CPU runtime is enough** unless the cache is missing.

In [ ]:
!pip -q install transformers scikit-learn scipy
import os
try:                                   # Google Colab: data and results on Google Drive
    from google.colab import drive
    drive.mount('/content/drive'); IN_COLAB = True
except ImportError:                    # local run from the notebooks/ folder
    IN_COLAB = False
DATA_DIR = os.environ.get("LINKEDIN_DATA_DIR",
    "/content/drive/MyDrive/fake_profile_paper" if IN_COLAB else os.path.abspath("../data"))
OUT_DIR = os.environ.get("LINKEDIN_OUT_DIR",
    "/content/drive/MyDrive/fake_profile_paper/linkedin_repro" if IN_COLAB else os.path.abspath("../results/run"))
print("data:", DATA_DIR, "| results:", OUT_DIR)

import os, ast, time, logging
import numpy as np, pandas as pd, torch, joblib
from transformers import RobertaTokenizer, RobertaModel
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import (accuracy_score, precision_score, recall_score, f1_score,
                             roc_auc_score, confusion_matrix, classification_report)

logging.basicConfig(level=logging.INFO, format='%(asctime)s - %(levelname)s - %(message)s')
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f"Using device: {device}")
RANDOM_STATE = 42
os.makedirs('embeddings_output', exist_ok=True)
os.makedirs('saved_models', exist_ok=True)

# ==== CELL ====
import json, math, random, itertools, warnings
from sklearn.metrics import average_precision_score, matthews_corrcoef, balanced_accuracy_score
warnings.filterwarnings("ignore")
os.makedirs(OUT_DIR, exist_ok=True)

# ---------------- experiment switches ----------------
SEEDS            = [42, 7, 123]      # 3 seeds ~ comfortable in one session; add 2024, 99 for 5
RUN_TASKS        = ["T1", "T1b", "T2"] # add "T3" if wanted
RUN_RANDOM       = True              # matched-budget random search
RUN_GRID         = True              # exhaustive 336-model search (first seed only; slow)
RUN_ABLATIONS    = True              # tag correction off / Full Name excluded (first seed)
FAITHFUL_REPLAY  = False             # already verified in the first run
HGS_KW = dict(pop_size=12, max_iter=18, early_stop_patience=4)   # as in the original run

In [ ]:
def load_raw():
    csv = os.path.join(DATA_DIR, "LinkedIn_Dataset.csv")
    pcl = os.path.join(DATA_DIR, "LinkedIn_Dataset.pcl")
    if os.path.exists(csv):
        df = pd.read_csv(csv); col = "test" if "test" in df.columns else "Label"
    elif os.path.exists(pcl):
        df = pd.read_pickle(pcl); col = "Label" if "Label" in df.columns else "test"
    else:
        raise FileNotFoundError("put LinkedIn_Dataset.csv or .pcl in " + DATA_DIR)
    code = pd.to_numeric(df[col], errors="coerce").astype(int).values
    return df.reset_index(drop=True), code

import hashlib
OFFICIAL_SHA256 = "6d67c8015fceee06226491a8b0683bfc01e220416fed99e7f2d91aca27ba2e1a"   # LinkedIn_Dataset.pcl, github.com/navid-aub/LinkedIn-Dataset
_p = os.path.join(DATA_DIR, "LinkedIn_Dataset.pcl")
if os.path.exists(os.path.join(DATA_DIR, "LinkedIn_Dataset.csv")):
    print("note: LinkedIn_Dataset.csv is present and is used instead of the official .pcl file")
elif os.path.exists(_p):
    _h = hashlib.sha256(open(_p, "rb").read()).hexdigest()
    print("dataset checksum:", "matches the official release" if _h == OFFICIAL_SHA256
          else "WARNING - differs from the official release (%s)" % _h)

dataset, RAW_CODE = load_raw()
counts = pd.Series(RAW_CODE).value_counts().sort_index().to_dict()
print("rows:", len(dataset), "| raw code counts:", counts)
assert counts == {0: 1800, 1: 600, 10: 600, 11: 600}, "counts differ from the README - check the file"
print("OK: matches the README (0=LLP, 1=FLP, 10/11=ChatGPT-generated)")

In [ ]:
TASKS = {
  "T1":  dict(neg=[0],    pos=[1],        balance_neg=None, name="legitimate vs manual fake (Gulati protocol)"),
  "T1b": dict(neg=[0],    pos=[1],        balance_neg=600,  name="legitimate vs manual fake (Ayoobi, balanced)"),
  "T2":  dict(neg=[0, 1], pos=[10, 11],   balance_neg=None, name="human-made vs ChatGPT-generated (original headline)"),
  "T3":  dict(neg=[0],    pos=[1, 10, 11],balance_neg=None, name="legitimate vs any fake"),
}
def task_index(task, seed):
    t = TASKS[task]
    neg = np.where(np.isin(RAW_CODE, t["neg"]))[0]
    pos = np.where(np.isin(RAW_CODE, t["pos"]))[0]
    if t["balance_neg"]:
        neg = np.random.default_rng(seed).choice(neg, t["balance_neg"], replace=False)
    idx = np.sort(np.r_[neg, pos])
    y = np.isin(RAW_CODE[idx], t["pos"]).astype(int)
    return idx, y
for k in RUN_TASKS:
    idx, y = task_index(k, 42)
    print("%-4s %-52s n=%-5d positives=%-5d (%.1f%%)" % (k, TASKS[k]["name"], len(y), y.sum(), 100*y.mean()))

In [ ]:
REQUIRED_COLUMNS = [
    'Intro','Full Name','Workplace','Location','Connections','Photo','Followers','About',
    'Experiences','Number of Experiences','Educations','Number of Educations','Licenses',
    'Number of Licenses','Volunteering','Number of Volunteering','Skills','Number of Skills',
    'Recommendations','Number of Recommendations','Projects','Number of Projects',
    'Publications','Number of Publications','Courses','Number of Courses','Honors',
    'Number of Honors','Scores','Number of Scores','Languages','Number of Languages',
    'Organizations','Number of Organizations','Interests','Number of Interests',
    'Activities','Number of Activities','Label']

def get_dtype(col):
    if col == 'Label': return 'Int64'
    if 'Number of' in col: return 'Int64'
    return 'string'

def parse_chatgpt_content(content):
    parts = content.split('#@')
    if len(parts) >= 4:
        return {'Full Name': parts[0].strip(),
                'Workplace': f"{parts[1].strip()} at {parts[2].strip()}",
                'Location':  parts[3].strip(),
                'Connections': '500+', 'Photo': 'No', 'Followers': ''}
    return {k: '' for k in ['Full Name','Workplace','Location','Connections','Photo','Followers']}

def clean_chatgpt_field(field, name):
    try:
        if isinstance(field, str):
            field = ast.literal_eval(field)
        if isinstance(field, dict) and 'chatGPT' in field:
            content = field['chatGPT']
            first = content.split('\n')[0]
            if first[:2] in ('1.','2.','3.','4.','5.'):
                first = first.split('.', 1)[1].strip()
            if name == 'Intro': return parse_chatgpt_content(first)
            if name == 'Full Name': return first.split('#@')[0].strip()
            if name == 'Workplace':
                p = first.split('#@')
                return f"{p[1].strip()} at {p[2].strip()}" if len(p) > 2 else ''
            if name == 'Location':
                p = first.split('#@')
                return p[3].strip() if len(p) > 3 else ''
            return first.replace('#@', ' ')
        if isinstance(field, dict): return field
    except Exception:
        return parse_chatgpt_content('') if name == 'Intro' else ''
    return field if field is not None else ''

def safe_eval(v):
    if isinstance(v, str):
        try: return ast.literal_eval(v)
        except Exception: return v
    return v

def to_float(v):
    try: return float(v)
    except Exception: return 0.0

NUMERICAL_FIELDS = ['Number of Experiences','Number of Educations','Number of Licenses',
                    'Number of Volunteering','Number of Skills','Number of Recommendations',
                    'Number of Projects','Number of Publications','Number of Courses',
                    'Number of Honors','Number of Scores','Number of Languages',
                    'Number of Organizations','Number of Interests','Number of Activities']
NUMERICAL_FIELD_NAMES = NUMERICAL_FIELDS

def clean_dataset(df):
    cleaned = pd.DataFrame(columns=REQUIRED_COLUMNS)
    for c in REQUIRED_COLUMNS:
        cleaned[c] = pd.Series(dtype=get_dtype(c))
    for c in df.columns:
        if c in REQUIRED_COLUMNS and c != 'Label':
            cleaned[c] = df[c]
    if 'Label' in df.columns:
        cleaned['Label'] = pd.to_numeric(df['Label'], errors='coerce').fillna(0).astype(int)
    else:
        # Définition de label : test ≥ 10 → 1 sinon 0 (cohérent avec notebook CatBoost LinkedIn)
        if 'test' in df.columns:
            cleaned['Label'] = df['test'].apply(lambda v: 1 if to_float(v) >= 10 else 0).astype(int)
        else:
            cleaned['Label'] = 0
    for c in ('Intro','Full Name','Workplace','Location'):
        if c in cleaned.columns:
            cleaned[c] = cleaned[c].apply(lambda v: clean_chatgpt_field(v, c))
    # Conditional imputation depuis Intro
    intros = cleaned['Intro'].apply(lambda v: v if isinstance(v, dict) else parse_chatgpt_content(''))
    for k in ('Full Name','Workplace','Location','Connections','Photo','Followers'):
        mask = cleaned[k].astype(str).str.strip().eq('') | cleaned[k].isna()
        cleaned.loc[mask, k] = intros[mask].apply(lambda d: d.get(k, ''))
    cleaned = cleaned.fillna('')
    return cleaned


cleaned_df = clean_dataset(dataset)
print("Cleaned:", cleaned_df.shape)

In [ ]:
tokenizer = RobertaTokenizer.from_pretrained('roberta-base')
roberta   = RobertaModel.from_pretrained('roberta-base').to(device).eval()

@torch.no_grad()
def embed_text(text):
    if not text or not text.strip():
        return np.zeros(roberta.config.hidden_size, dtype=np.float32)
    inp = tokenizer(text, return_tensors='pt', padding=True,
                    truncation=True, max_length=512).to(device)
    out = roberta(**inp).last_hidden_state.mean(dim=1).squeeze().cpu().numpy()
    return out.astype(np.float32)

def section_text_embedding(content, tag):
    return embed_text(content) - embed_text(tag)


def section_texts(df, include_name=True):
    rows = []
    for i in range(len(df)):
        intro = safe_eval(df['Intro'].iloc[i]); about = safe_eval(df['About'].iloc[i])
        exps  = safe_eval(df['Experiences'].iloc[i])
        if isinstance(intro, dict):
            parts = ([intro.get('Full Name','')] if include_name else []) + \
                    [intro.get('Workplace',''), intro.get('Location','')]
            intro_text = " ".join(str(p) for p in parts)
        else:
            intro_text = str(intro)
        about_text = about.get('About','') if isinstance(about, dict) else str(about)
        exp_text = ""
        if isinstance(exps, dict): exps = exps.values()
        if isinstance(exps, (list, type({}.values()))):
            for e in exps:
                if isinstance(e, dict):
                    exp_text += " ".join(str(e.get(k,'')) for k in
                        ['Role','Workplace','Duration','Workplace Location','Description']) + " "
        rows.append((intro_text, about_text, exp_text))
    return rows

def cached_section_embeddings(include_name):
    path = os.path.join(OUT_DIR, "roberta_sections_name%d.npy" % int(include_name))
    if os.path.exists(path):
        return np.load(path)
    if not torch.cuda.is_available():
        raise RuntimeError('No cached embeddings found at ' + path + ' and no GPU. '
                           'Switch the runtime to GPU T4 for the first run.')
    rows = section_texts(cleaned_df, include_name)
    E = np.zeros((len(rows), 3, roberta.config.hidden_size), dtype=np.float32)
    for i, (a, b, c) in enumerate(rows):
        E[i, 0], E[i, 1], E[i, 2] = embed_text(a), embed_text(b), embed_text(c)
        if (i+1) % 300 == 0: print("  embedded", i+1, "/", len(rows))
    np.save(path, E); return E

TAG_E = np.stack([embed_text("Introduction"), embed_text("About"), embed_text("Experiences")])
SEC_NAME   = cached_section_embeddings(include_name=True)    # original behaviour
try:
    SEC_NONAME = cached_section_embeddings(include_name=False)   # cached by notebook 00
except Exception as e:
    SEC_NONAME = None; print('no-name embeddings not available:', e)

def text_features(sec, tag_correction=True):
    return ((sec - TAG_E[None]) if tag_correction else sec).mean(axis=1)

NUM_X = np.array([[to_float(cleaned_df[f].iloc[i]) for f in NUMERICAL_FIELDS]
                  for i in range(len(cleaned_df))], dtype=np.float32)
print("sections:", SEC_NAME.shape, "| numeric:", NUM_X.shape)

In [ ]:
# =============================================================================
#  MODULE : Hunger Games Search (HGS) + Fuzzy OWA Aggregation Anti-Overfitting
#  ---------------------------------------------------------------------------
#  Référence HGS : Yang Y., Chen H., Heidari A.A., Gandomi A.H. (2021)
#                  "Hunger Games Search", ESWA 177, 114864.
#  Référence OWA  : Yager R.R. (1988) "On Ordered Weighted Averaging Aggregation
#                                      Operators in Multi-Criteria Decision Making"
#
#  STRATÉGIE ANTI-OVERFITTING :
#  ---------------------------------------------------------------------------
#  1)  Le test set n'est JAMAIS vu par le fitness (sacré jusqu'à l'évaluation
#      finale). Fitness = K-fold CV stratifiée sur (X_train, y_train).
#  2)  Agrégation FLOUE (OWA pessimiste) des K scores : on donne plus de poids
#      aux PIRES plis → pénalise la variance entre folds → robustesse.
#  3)  Fuzzy multi-metric : fusion F1 ⊕ AUC ⊕ Recall par OWA pondéré (chaque
#      métrique est min-max normalisée).
#  4)  Pénalité de généralisation : fitness final = OWA_cv - λ · (train - μ_cv)
#      → décourage les modèles qui sur-apprennent (gap train-CV élevé).
#  5)  Early-stopping HGS : si pas d'amélioration pendant `patience` itérations,
#      on arrête → moins d'évaluations, moins de risque de surajustement à un
#      pli particulier.
# =============================================================================
import math, random, time
import numpy as np
from sklearn.model_selection import StratifiedKFold
from sklearn.base import clone
from sklearn.metrics import f1_score, roc_auc_score, recall_score, accuracy_score


# -----------------------------------------------------------------------------
#  1. Fuzzy OWA (Ordered Weighted Average) — Yager 1988
# -----------------------------------------------------------------------------
def owa_weights(n, orness=0.30):
    """
    Génère des poids OWA pour un vecteur de taille n.
        orness ∈ [0,1] :
            • 0.0  → "AND" (min)              très pessimiste
            • 0.5  → moyenne arithmétique
            • 1.0  → "OR"  (max)              très optimiste
        On choisit orness = 0.30 (légèrement pessimiste) : on récompense
        les modèles dont les PIRES folds restent corrects.
    Implémentation : poids exponentiels biaisés vers la fin (queue basse).
    """
    if n <= 1:
        return np.ones(n) / max(n, 1)
    # Distribution exponentielle : alpha contrôle la pente
    # alpha = -ln(orness)  ⇒ pour orness < 0.5, poids concentrés sur les PETITES
    #                       valeurs (premiers éléments du vecteur trié ascending)
    #                       → moyenne pondérée tirée vers le bas = pessimiste.
    alpha = max(-math.log(max(orness, 1e-6)), 1e-6)
    idx = np.arange(n)
    # poids décroissants : w[0] (pire valeur) = max, w[n-1] (meilleure) = min
    w = np.exp(-alpha * idx / max(n - 1, 1))
    return w / w.sum()


def fuzzy_owa(scores, orness=0.30):
    """
    Agrégation OWA d'un vecteur de scores.
    1) Trier les scores en ordre croissant.
    2) Appliquer w_i tel que les premiers (pires) reçoivent plus de poids.
    """
    arr = np.asarray(scores, dtype=float).ravel()
    if arr.size == 0:
        return 0.0
    arr = np.sort(arr)                          # ordre croissant
    w   = owa_weights(arr.size, orness=orness)  # poids biaisés bas
    return float(np.dot(w, arr))


def fuzzy_multi_metric(y_true, y_pred, y_proba=None, orness=0.30):
    """
    Combine F1, Recall et AUC (si y_proba) en un score flou unique via OWA.
    Chaque métrique ∈ [0,1] → comparable.
    """
    binary = len(np.unique(y_true)) == 2
    avg = "binary" if binary else "macro"
    f1  = f1_score(y_true, y_pred, average=avg, zero_division=0)
    rec = recall_score(y_true, y_pred, average=avg, zero_division=0)
    metrics = [f1, rec]
    if y_proba is not None and binary:
        try:
            auc = roc_auc_score(y_true, y_proba)
            metrics.append(auc)
        except Exception:
            pass
    return fuzzy_owa(metrics, orness=orness)


# -----------------------------------------------------------------------------
#  2. Fitness K-fold CV avec fuzzy aggregation + pénalité de gap
# -----------------------------------------------------------------------------
def make_cv_fitness(estimator_factory, X_train, y_train,
                    n_splits=5, seed=42, orness=0.30,
                    gap_penalty=0.50, use_proba=True):
    """
    Renvoie une fonction de fitness `fit(params) -> float` agrégée via OWA flou.

        estimator_factory : callable(params:dict) -> sklearn-compatible model
        n_splits          : nb de folds (5 par défaut)
        orness            : ∈ [0,1], <0.5 = pessimiste, pénalise variance
        gap_penalty       : λ, pénalité du gap (train - μ_cv) ∈ [0,1]
        use_proba         : True si l'estimator expose predict_proba (AUC)

    Le SCORE final renvoyé :
        score = OWA_cv  -  λ · max(0, train_full - mean_cv)
    où :
        OWA_cv     = agrégation floue OWA des K scores fuzzy_multi_metric
        train_full = score fuzzy_multi_metric sur l'ENSEMBLE train re-fit
        gap        = (train_full - mean_cv)  → quantifie l'overfitting
    """
    X_train = np.asarray(X_train) if not hasattr(X_train, "toarray") else X_train
    y_train = np.asarray(y_train)

    def _safe_score(est, X_tr, y_tr, X_va, y_va):
        try:
            est.fit(X_tr, y_tr)
            y_pred = est.predict(X_va)
            y_proba = None
            if use_proba and hasattr(est, "predict_proba"):
                try:
                    y_proba = est.predict_proba(X_va)[:, 1]
                except Exception:
                    y_proba = None
            return fuzzy_multi_metric(y_va, y_pred, y_proba, orness=orness)
        except Exception:
            return 0.0

    def fitness(params):
        # --- K-fold CV stratifiée sur X_train UNIQUEMENT (jamais X_test) ---
        skf = StratifiedKFold(n_splits=n_splits, shuffle=True, random_state=seed)
        fold_scores = []
        try:
            est_template = estimator_factory(params)
        except Exception:
            return 0.0

        for tr_idx, va_idx in skf.split(np.zeros(len(y_train)), y_train):
            # sparse-safe indexing
            if hasattr(X_train, "tocsr"):
                Xtr_f, Xva_f = X_train.tocsr()[tr_idx], X_train.tocsr()[va_idx]
            else:
                Xtr_f, Xva_f = X_train[tr_idx], X_train[va_idx]
            ytr_f, yva_f = y_train[tr_idx], y_train[va_idx]
            est = clone(est_template)
            fold_scores.append(_safe_score(est, Xtr_f, ytr_f, Xva_f, yva_f))

        owa_cv  = fuzzy_owa(fold_scores, orness=orness)
        mean_cv = float(np.mean(fold_scores))

        # --- Score train complet pour calcul de gap (overfitting) ---
        est_full = clone(est_template)
        try:
            est_full.fit(X_train, y_train)
            y_pred_tr = est_full.predict(X_train)
            y_proba_tr = None
            if use_proba and hasattr(est_full, "predict_proba"):
                try:
                    y_proba_tr = est_full.predict_proba(X_train)[:, 1]
                except Exception:
                    y_proba_tr = None
            train_full = fuzzy_multi_metric(y_train, y_pred_tr, y_proba_tr, orness=orness)
        except Exception:
            train_full = mean_cv

        gap     = max(0.0, train_full - mean_cv)
        penalty = gap_penalty * gap
        return owa_cv - penalty

    return fitness


# -----------------------------------------------------------------------------
#  3. HGS Optimizer (avec early-stopping pour limiter le risque d'overfit)
# -----------------------------------------------------------------------------
class HGSOptimizer:
    """
    Hunger Games Search — Yang et al. 2021.
    Compatible recherche discrète : search_space = {param: [v1, v2, ...]}.

    Améliorations anti-overfitting :
        • early_stop_patience : arrête HGS si pas d'amélioration sur N itérations
        • Le fitness est typiquement obtenu via make_cv_fitness (fuzzy OWA + gap).
    """
    def __init__(self, search_space, fitness_fn,
                 pop_size=8, max_iter=15,
                 l_h=1e-2, LH=1e4,
                 seed=42, verbose=True,
                 early_stop_patience=4):
        self.space   = search_space
        self.keys    = list(search_space.keys())
        self.dims    = [len(search_space[k]) for k in self.keys]
        self.D       = len(self.keys)
        self.fit     = fitness_fn
        self.N       = pop_size
        self.T       = max_iter
        self.l_h     = l_h
        self.LH      = LH
        self.verbose = verbose
        self.patience = early_stop_patience
        random.seed(seed); np.random.seed(seed)
        self.history = []
        self.best_params_ = None
        self.best_score_  = -np.inf

    def _decode(self, x):
        params = {}
        for i, k in enumerate(self.keys):
            idx = int(min(self.dims[i] - 1, max(0, math.floor(x[i] * self.dims[i]))))
            params[k] = self.space[k][idx]
        return params

    def _evaluate(self, x, cache):
        params = self._decode(x)
        key = tuple(sorted(params.items(), key=lambda kv: kv[0]))
        if key in cache:
            return cache[key], params
        s = self.fit(params)
        cache[key] = s
        return s, params

    def optimize(self):
        X = np.random.rand(self.N, self.D)
        F = np.full(self.N, -np.inf)
        Hunger = np.zeros(self.N)
        cache = {}

        for i in range(self.N):
            F[i], _ = self._evaluate(X[i], cache)

        best_idx = int(np.argmax(F))
        X_best   = X[best_idx].copy()
        F_best   = F[best_idx]
        self.best_params_ = self._decode(X_best)
        self.best_score_  = F_best
        self.history.append(F_best)

        if self.verbose:
            print(f"[HGS] init  best fuzzy-score={F_best:.4f}  params={self.best_params_}")

        no_improve = 0
        for t in range(self.T):
            E_t = 2 * (1 - t / max(1, self.T))
            best_fit  = F.max();  worst_fit = F.min()

            for i in range(self.N):
                if abs(best_fit - worst_fit) < 1e-12:
                    TH = 0
                else:
                    TH = (F[i] - best_fit) / (best_fit - worst_fit + 1e-12) * \
                         random.random() * 2 * (1 - t / max(1, self.T))
                hunger_i = TH if abs(TH) > self.l_h else self.l_h
                Hunger[i] += min(self.LH, abs(hunger_i))

            S_hunger = Hunger.sum() + 1e-12
            W1 = np.zeros(self.N); W2 = np.zeros(self.N)
            for i in range(self.N):
                if random.random() < (Hunger[i] / S_hunger):
                    W1[i] = Hunger[i] * self.N / S_hunger * random.random()
                else:
                    W1[i] = 1.0
                W2[i] = (1 - math.exp(-abs(Hunger[i] - Hunger[best_idx]))) * \
                        random.random() * 2

            improved_iter = False
            for i in range(self.N):
                r1, r2 = random.random(), random.random()
                a = 2 * E_t * r1 - E_t
                R = a
                E = math.copysign(1, random.random() - 0.5) * \
                    (math.exp(-abs(F[i] - F_best) / (abs(F_best) + 1e-12)))

                levy_step = 0.01 * np.random.standard_cauchy(self.D)
                if r1 < self.l_h:
                    X_new = np.random.rand(self.D)
                elif r2 > E:
                    X_new = W1[i] * X_best + R * W2[i] * abs(X_best - X[i])
                else:
                    X_new = W1[i] * X_best - R * W2[i] * abs(X_best - X[i]) + levy_step
                X_new = np.clip(X_new, 0.0, 1.0)

                F_new, params_new = self._evaluate(X_new, cache)
                if F_new > F[i]:
                    X[i] = X_new
                    F[i] = F_new
                    if F_new > F_best:
                        F_best = F_new
                        X_best = X_new.copy()
                        best_idx = i
                        self.best_params_ = params_new
                        self.best_score_  = F_best
                        improved_iter = True
                        if self.verbose:
                            print(f"[HGS] t={t+1:02d}  ↑  fuzzy-score={F_best:.4f}  {params_new}")

            self.history.append(F_best)
            no_improve = 0 if improved_iter else (no_improve + 1)
            if no_improve >= self.patience:
                if self.verbose:
                    print(f"[HGS] early-stop @ t={t+1} (no improvement for {self.patience} iters)")
                break

        if self.verbose:
            print(f"[HGS] DONE   best fuzzy-score={self.best_score_:.4f}  "
                  f"params={self.best_params_}")
        return self.best_params_, self.best_score_, self.history


def hgs_search(search_space, fitness_fn,
               pop_size=8, max_iter=15, seed=42, verbose=True,
               early_stop_patience=4):
    """API simple : best_params, best_score, history = hgs_search(...)"""
    opt = HGSOptimizer(search_space, fitness_fn,
                       pop_size=pop_size, max_iter=max_iter,
                       seed=seed, verbose=verbose,
                       early_stop_patience=early_stop_patience)
    t0 = time.time()
    best_params, best_score, history = opt.optimize()
    print(f"[HGS] elapsed = {time.time()-t0:.1f}s")
    return best_params, best_score, history

In [ ]:
LR_SPACE = {
    "C":            [0.01, 0.05, 0.1, 0.3, 0.5, 1.0, 2.0],   # plafond abaissé
    "penalty":      ["l1", "l2", "elasticnet"],
    "solver":       ["liblinear", "saga", "lbfgs"],
    "class_weight": [None, "balanced"],
    "max_iter":     [500, 1000, 2000],
    "l1_ratio":     [0.1, 0.5, 0.9],
}

VALID_COMBOS = {
    ("l1", "liblinear"), ("l1", "saga"),
    ("l2", "liblinear"), ("l2", "saga"), ("l2", "lbfgs"),
    ("elasticnet", "saga"),
}

def build_estimator(params):
    p = dict(params)
    if (p["penalty"], p["solver"]) not in VALID_COMBOS:
        # combinaison invalide : on retombe sur (l2, lbfgs)
        p["penalty"], p["solver"] = "l2", "lbfgs"
    if p["penalty"] != "elasticnet":
        p.pop("l1_ratio", None)
    if p["solver"] == "liblinear":
        return LogisticRegression(**p, random_state=42, n_jobs=1)
    return LogisticRegression(**p, random_state=42, n_jobs=-1)


def canonical(params):
    # the model build_estimator() actually fits - used to merge duplicate grid points
    p = dict(params)
    if (p["penalty"], p["solver"]) not in VALID_COMBOS: p["penalty"], p["solver"] = "l2", "lbfgs"
    if p["penalty"] != "elasticnet": p["l1_ratio"] = None
    return tuple(sorted(p.items(), key=lambda kv: kv[0]))

DISTINCT = {}
for combo in itertools.product(*LR_SPACE.values()):
    p = dict(zip(LR_SPACE.keys(), combo)); DISTINCT.setdefault(canonical(p), p)
print("nominal grid:", int(np.prod([len(v) for v in LR_SPACE.values()])),
      "| distinct models:", len(DISTINCT))

In [ ]:
class Counted:
    # wraps a fitness so the number of DISTINCT models evaluated is known exactly
    def __init__(self, fn): self.fn, self.seen = fn, {}
    def __call__(self, params):
        k = canonical(params)
        if k not in self.seen: self.seen[k] = self.fn(params)
        return self.seen[k]
    @property
    def n(self): return len(self.seen)

def random_search(fitness, budget, seed):
    keys = list(DISTINCT.keys()); rng = random.Random(seed); rng.shuffle(keys)
    best_p, best_s = None, -1e9
    for k in keys[:budget]:
        s = fitness(DISTINCT[k])
        if s > best_s: best_p, best_s = DISTINCT[k], s
    return best_p, best_s

def grid_search(fitness):
    best_p, best_s = None, -1e9
    for i, p in enumerate(DISTINCT.values()):
        s = fitness(p)
        if s > best_s: best_p, best_s = p, s
        if (i+1) % 50 == 0: print("    grid", i+1, "/", len(DISTINCT), "best", round(best_s, 4))
    return best_p, best_s

def evaluate(y, proba, thr=0.5):
    yp = (proba >= thr).astype(int)
    tn, fp, fn, tp = confusion_matrix(y, yp, labels=[0,1]).ravel()
    return dict(
        accuracy=accuracy_score(y, yp),
        precision_pos=precision_score(y, yp, zero_division=0),
        recall_pos=recall_score(y, yp, zero_division=0),
        f1_pos=f1_score(y, yp, zero_division=0),
        f1_macro=f1_score(y, yp, average="macro", zero_division=0),
        precision_w=precision_score(y, yp, average="weighted", zero_division=0),
        recall_w=recall_score(y, yp, average="weighted", zero_division=0),
        f1_w=f1_score(y, yp, average="weighted", zero_division=0),
        roc_auc=roc_auc_score(y, proba), pr_auc=average_precision_score(y, proba),
        mcc=matthews_corrcoef(y, yp), bal_acc=balanced_accuracy_score(y, yp),
        TP=int(tp), FP=int(fp), FN=int(fn), TN=int(tn), n_test=int(len(y)))

## Autosave and resume

- Every model evaluation is cached to Drive the moment it finishes. HGS and random search are
  deterministic for a given seed, so after a disconnect a re-run replays the cached part instantly
  and continues from the exact model where it stopped - including midway through a grid.
- Every finished run is appended to this job's own CSV. Runs already in the CSV are skipped.
- T3 is split with stratification on the raw 4-way code, which reproduces Gulati et al.'s
  GPT-3.5 protocol exactly (1,260/420/840 train, 540/180/360 test). All other tasks keep the
  original stratification on the binary label, so their splits match the first run.

In [ ]:
import pickle
CACHE_DIR = os.path.join(OUT_DIR, "cache"); os.makedirs(CACHE_DIR, exist_ok=True)

def _atomic_pickle(path, obj):
    tmp = path + ".tmp"
    with open(tmp, "wb") as f: pickle.dump(obj, f)
    os.replace(tmp, path)

def _atomic_csv(df, path):
    tmp = path + ".tmp"; df.to_csv(tmp, index=False); os.replace(tmp, path)

class PersistentCounted:
    # fitness wrapper: saves each evaluation to Drive; n = distinct models queried by THIS search
    def __init__(self, fn, cache_path):
        self.fn, self.path = fn, cache_path
        self.cache = pickle.load(open(cache_path, "rb")) if os.path.exists(cache_path) else {}
        self.queried = set()
        if self.cache: print("    resuming from %d cached evaluations" % len(self.cache))
    def __call__(self, params):
        k = canonical(params); self.queried.add(k)
        if k not in self.cache:
            self.cache[k] = self.fn(params); _atomic_pickle(self.path, self.cache)
        return self.cache[k]
    @property
    def n(self): return len(self.queried)

def prepare(task, seed, sec=None, tag_correction=True):
    idx, y = task_index(task, seed)
    sec = SEC_NAME if sec is None else sec
    X = np.hstack([text_features(sec[idx], tag_correction), NUM_X[idx]]).astype(np.float32)
    strat = RAW_CODE[idx] if task == "T3" else y
    Xtr, Xte, ytr, yte = train_test_split(X, y, test_size=0.30, random_state=seed, stratify=strat)
    sc = StandardScaler().fit(Xtr)                              # training split only
    return sc.transform(Xtr), sc.transform(Xte), ytr, yte

def run_saved(job, task, seed, strategy, variant="main", budget=None, sec=None, tag_correction=True):
    out = os.path.join(OUT_DIR, "results_%s.csv" % job)
    done = pd.read_csv(out) if os.path.exists(out) else pd.DataFrame()
    if len(done):
        hit = done[(done.task == task) & (done.seed == seed) &
                   (done.strategy == strategy) & (done.variant == variant)]
        if len(hit):
            print("%-4s seed %-4d %-7s %-18s already done - skipped" % (task, seed, strategy, variant))
            return hit.iloc[0].to_dict()
    Xtr, Xte, ytr, yte = prepare(task, seed, sec, tag_correction)
    cache = os.path.join(CACHE_DIR, "%s__%s_s%d_%s.pkl" % (job, task, seed, variant))
    fit = PersistentCounted(make_cv_fitness(build_estimator, Xtr, ytr, n_splits=5, seed=seed,
                                            orness=0.30, gap_penalty=0.50, use_proba=True), cache)
    t0 = time.time()
    if strategy == "HGS":
        bp, bs, _ = hgs_search(LR_SPACE, fit, seed=seed, verbose=False, **HGS_KW)
    elif strategy == "Random":
        bp, bs = random_search(fit, int(budget), seed)
    else:
        bp, bs = grid_search(fit)
    m = build_estimator(bp).fit(Xtr, ytr)
    r = evaluate(yte, m.predict_proba(Xte)[:, 1])
    r.update(job=job, task=task, seed=seed, strategy=strategy, variant=variant, cv_fitness=bs,
             n_models=fit.n, seconds=round(time.time() - t0, 1), params=json.dumps(bp),
             params_fitted=json.dumps(dict(canonical(bp))))
    _atomic_csv(pd.concat([done, pd.DataFrame([r])], ignore_index=True), out)
    print("%-4s seed %-4d %-7s %-18s models=%-3d F1+=%.4f acc=%.4f  (saved to %s)"
          % (task, seed, strategy, variant, r["n_models"], r["f1_pos"], r["accuracy"], os.path.basename(out)))
    return r

## Embedding variants (as in notebooks 07 and 09)

In [ ]:
from sklearn.model_selection import StratifiedKFold
from sklearn.linear_model import LogisticRegression

def _unit(x):
    return x / (np.linalg.norm(x, axis=-1, keepdims=True) + 1e-12)

def _r2(E, direction):
    # Ren et al. R2: remove the component along `direction`, then normalise to unit length
    d = _unit(direction)
    return _unit(E - (E * d).sum(-1, keepdims=True) * d)

def _coherence(E):
    En = _unit(E)
    c = np.stack([(En[:, 0] * En[:, 1]).sum(1), (En[:, 0] * En[:, 2]).sum(1),
                  (En[:, 1] * En[:, 2]).sum(1)], 1)
    return np.c_[c, c.mean(1), c.min(1), c.max(1), c.std(1)]

def _l1(E):
    return np.stack([np.abs(E[:, 0] - E[:, 1]).sum(1), np.abs(E[:, 0] - E[:, 2]).sum(1),
                     np.abs(E[:, 1] - E[:, 2]).sum(1)], 1)

def embed_variant(v, Etr, Ete):
    # Etr, Ete: (n, 3, 768) raw section embeddings. Statistics come from Etr only.
    mu   = Etr.reshape(-1, Etr.shape[-1]).mean(0)          # global training mean
    mu_s = Etr.mean(0)                                     # per-section training means (3, 768)
    T    = TAG_E[None]                                     # tag embeddings (1, 3, 768)
    def f(E):
        if v == "V0": return (E - T).mean(1)
        if v == "V1": return E.mean(1)
        if v == "V2": return _unit(E - mu).mean(1)
        if v == "V3": return _r2(E, mu).mean(1)
        if v == "V4": return _r2(E, mu_s[None]).mean(1)
        if v == "V5": return _r2(E, T).mean(1)
        if v == "V7": return np.c_[(E - T).mean(1), _coherence(E)]
        if v == "V8":
            R = _r2(E, mu_s[None]); return np.c_[R.mean(1), _coherence(R)]
        if v == "V9": return np.c_[(E - T).mean(1), _l1(E)]
        raise ValueError(v)
    if v == "V6":
        A = Etr.reshape(-1, Etr.shape[-1]) - mu
        lam, U = np.linalg.eigh(np.cov(A, rowvar=False))
        lam = np.clip(lam, 0, None); eps = 1e-2 * lam.mean()   # shrinkage for the small eigenvalues
        W = U @ np.diag(1.0 / np.sqrt(lam + eps)) @ U.T
        return ((Etr - mu) @ W).mean(1), ((Ete - mu) @ W).mean(1)
    return f(Etr), f(Ete)

VARIANT_IDS = ["V0", "V1", "V2", "V3", "V4", "V5", "V6", "V7", "V8", "V9"]
C_GRID = [0.1, 0.5, 1.0]

def fit_screen(Xtr, ytr, seed):
    skf = StratifiedKFold(n_splits=3, shuffle=True, random_state=seed)
    best_c, best_f = C_GRID[0], -1
    for c in C_GRID:
        fs = []
        for a, b in skf.split(Xtr, ytr):
            m = LogisticRegression(penalty="l1", solver="liblinear", C=c, class_weight="balanced",
                                   max_iter=2000, random_state=seed).fit(Xtr[a], ytr[a])
            fs.append(f1_score(ytr[b], m.predict(Xtr[b]), zero_division=0))
        if np.mean(fs) > best_f: best_c, best_f = c, np.mean(fs)
    m = LogisticRegression(penalty="l1", solver="liblinear", C=best_c, class_weight="balanced",
                           max_iter=2000, random_state=seed).fit(Xtr, ytr)
    return m, best_c, best_f

EMB_OUT = os.path.join(OUT_DIR, "results_embeddings.csv")

def run_variant(v, task, seed):
    done = pd.read_csv(EMB_OUT) if os.path.exists(EMB_OUT) else pd.DataFrame()
    if len(done) and ((done.variant == v) & (done.task == task) & (done.seed == seed)).any():
        return None
    idx, y = task_index(task, seed)
    strat = RAW_CODE[idx] if task == "T3" else y
    tr, te = train_test_split(np.arange(len(idx)), test_size=0.30, random_state=seed, stratify=strat)
    Etr, Ete = SEC_NAME[idx[tr]], SEC_NAME[idx[te]]
    Ztr, Zte = embed_variant(v, Etr, Ete)
    Xtr = np.hstack([Ztr, NUM_X[idx[tr]]]); Xte = np.hstack([Zte, NUM_X[idx[te]]])
    sc = StandardScaler().fit(Xtr); Xtr, Xte = sc.transform(Xtr), sc.transform(Xte)
    t0 = time.time()
    m, c, cvf = fit_screen(Xtr, y[tr], seed)
    r = evaluate(y[te], m.predict_proba(Xte)[:, 1])
    r.update(variant=v, task=task, seed=seed, C=c, cv_f1=cvf, n_features=Xtr.shape[1],
             nonzero_coef=int((m.coef_ != 0).sum()), seconds=round(time.time() - t0, 1))
    _atomic_csv(pd.concat([done, pd.DataFrame([r])], ignore_index=True), EMB_OUT)
    print("%-4s %-4s seed %-4d C=%-4s F1+=%.4f AUC=%.4f MCC=%.4f" %
          (v, task, seed, c, r["f1_pos"], r["roc_auc"], r["mcc"]))
    return r

## Statistics and helpers

In [ ]:
# ---- statistics shared by notebooks 10-14 ----------------------------------------------------
from scipy import stats as sst

def _midrank(x):
    J = np.argsort(x, kind="mergesort"); Z = x[J]; N = len(x); T = np.zeros(N); i = 0
    while i < N:
        j = i
        while j < N and Z[j] == Z[i]:
            j += 1
        T[i:j] = 0.5 * (i + j - 1) + 1
        i = j
    out = np.empty(N); out[J] = T
    return out

def delong_test(y, s1, s2):
    """DeLong, DeLong and Clarke-Pearson (1988) test for two correlated ROC areas,
    computed with the fast algorithm of Sun and Xu (2014). Returns auc1, auc2, z, two-sided p."""
    y = np.asarray(y).astype(int); pos, neg = y == 1, y == 0; m, n = int(pos.sum()), int(neg.sum())
    aucs, V10, V01 = [], [], []
    for s in (np.asarray(s1, float), np.asarray(s2, float)):
        X, Y = s[pos], s[neg]
        tx, ty, tz = _midrank(X), _midrank(Y), _midrank(np.r_[X, Y])
        aucs.append((tz[:m].sum() - m * (m + 1) / 2) / (m * n))
        V10.append((tz[:m] - tx) / n); V01.append(1.0 - (tz[m:] - ty) / m)
    S10, S01 = np.cov(np.vstack(V10)), np.cov(np.vstack(V01))
    var = (S10[0, 0] + S10[1, 1] - 2 * S10[0, 1]) / m + (S01[0, 0] + S01[1, 1] - 2 * S01[0, 1]) / n
    if var <= 0:
        return aucs[0], aucs[1], 0.0, 1.0
    z = (aucs[0] - aucs[1]) / np.sqrt(var)
    return aucs[0], aucs[1], float(z), float(2 * sst.norm.sf(abs(z)))

def mcnemar_exact(y, pred1, pred2):
    """Exact McNemar test on the discordant pairs. Returns (only 1 right, only 2 right, two-sided p)."""
    y, pred1, pred2 = map(np.asarray, (y, pred1, pred2))
    n12 = int(((pred1 == y) & (pred2 != y)).sum()); n21 = int(((pred1 != y) & (pred2 == y)).sum())
    p = sst.binomtest(min(n12, n21), n12 + n21, 0.5).pvalue if n12 + n21 else 1.0
    return n12, n21, float(p)

def f1_pos_fast(y, p):
    tp = np.sum((p == 1) & (y == 1)); fp = np.sum((p == 1) & (y == 0)); fn = np.sum((p == 0) & (y == 1))
    return 2 * tp / max(2 * tp + fp + fn, 1)

def acc_fast(y, p):
    return float(np.mean(y == p))

def macro_f1_fast(y, p, k=None):
    k = int(max(y.max(), p.max()) + 1) if k is None else k
    cm = np.zeros((k, k)); np.add.at(cm, (y, p), 1)
    tp = np.diag(cm); prec = tp / np.maximum(cm.sum(0), 1); rec = tp / np.maximum(cm.sum(1), 1)
    return float(np.mean(2 * prec * rec / np.maximum(prec + rec, 1e-12)))

def _strat_resample(y, rng):
    return np.concatenate([rng.choice(np.where(y == c)[0], np.sum(y == c)) for c in np.unique(y)])

def bootstrap_ci(y, pred, metric, B=5000, seed=0):
    """Stratified bootstrap of the test set: point value, 95% percentile interval, standard error."""
    y, pred = np.asarray(y), np.asarray(pred); rng = np.random.default_rng(seed)
    v = np.array([metric(y[i], pred[i]) for i in (_strat_resample(y, rng) for _ in range(B))])
    return metric(y, pred), np.percentile(v, [2.5, 97.5]), float(v.std(ddof=1)), v

def paired_bootstrap(y, pred_a, pred_b, metric, B=5000, seed=0):
    """Difference metric(a) - metric(b) on the same test profiles, with a paired stratified bootstrap."""
    y, a, b = map(np.asarray, (y, pred_a, pred_b)); rng = np.random.default_rng(seed)
    d = np.array([metric(y[i], a[i]) - metric(y[i], b[i]) for i in (_strat_resample(y, rng) for _ in range(B))])
    return metric(y, a) - metric(y, b), np.percentile(d, [2.5, 97.5]), float(d.std(ddof=1))

def corrected_resampled_t(diffs, n_train, n_test, alternative="two-sided"):
    """Nadeau and Bengio (2003) corrected resampled t-test for J repeated random splits:
    the variance of the mean difference is inflated by (1/J + n_test/n_train) because training
    sets overlap between splits. Returns mean difference, t, p."""
    d = np.asarray(diffs, float); J = len(d)
    if J < 2 or np.allclose(d, d[0]):
        return float(d.mean()) if J else np.nan, np.nan, np.nan
    t = d.mean() / np.sqrt((1.0 / J + n_test / n_train) * d.var(ddof=1))
    p = sst.t.sf(t, J - 1) if alternative == "greater" else 2 * sst.t.sf(abs(t), J - 1)
    return float(d.mean()), float(t), float(p)

def wilcoxon_p(diffs, alternative="two-sided"):
    d = np.asarray(diffs, float)
    if np.all(d == 0) or len(d) < 2:
        return np.nan
    try:
        return float(sst.wilcoxon(d, alternative=alternative, zero_method="wilcox").pvalue)
    except ValueError:
        return np.nan

def sign_test_p(diffs, alternative="two-sided"):
    d = np.asarray(diffs, float); k, n = int((d > 0).sum()), int((d != 0).sum())
    return float(sst.binomtest(k, n, 0.5, alternative=alternative).pvalue) if n else np.nan

def seed_mean_ci(values, B=20000, seed=0):
    """Percentile interval for a mean over seeds, by resampling seeds (wide when there are few seeds)."""
    v = np.asarray(values, float); rng = np.random.default_rng(seed)
    return np.percentile(rng.choice(v, (B, len(v))).mean(1), [2.5, 97.5])

def min_detectable_difference(se_diff, alpha=0.05, power=0.80):
    """Smallest true difference a two-sided test at level alpha detects with the given power."""
    return (sst.norm.ppf(1 - alpha / 2) + sst.norm.ppf(power)) * se_diff

def fmt_p(p):
    return "n/a" if p is None or (isinstance(p, float) and np.isnan(p)) else ("<0.001" if p < 0.001 else "%.3f" % p)

In [ ]:
from sklearn.base import clone
PRED_DIR = os.path.join(OUT_DIR, "predictions"); os.makedirs(PRED_DIR, exist_ok=True)

def split_task(task, seed):
    """The exact partition of notebooks 00-09: stratified 70/30 (T3 stratified on the raw code)."""
    idx, y = task_index(task, seed)
    strat = RAW_CODE[idx] if task == "T3" else y
    tr, te = train_test_split(np.arange(len(idx)), test_size=0.30, random_state=seed, stratify=strat)
    return idx, y, tr, te

def features(variant, idx, tr, te, sec=None):
    """Embedding variant + 15 counts, standardised with training-split statistics only."""
    sec = SEC_NAME if sec is None else sec
    Ztr, Zte = embed_variant(variant, sec[idx[tr]], sec[idx[te]])
    Xtr = np.hstack([Ztr, NUM_X[idx[tr]]]); Xte = np.hstack([Zte, NUM_X[idx[te]]])
    sc = StandardScaler().fit(Xtr)
    return sc.transform(Xtr), sc.transform(Xte)

def save_pred(name, **arrays):
    np.savez_compressed(os.path.join(PRED_DIR, name + ".npz"), **arrays)

def load_pred(name):
    p = os.path.join(PRED_DIR, name + ".npz")
    return dict(np.load(p)) if os.path.exists(p) else None

def done_row(path, **key):
    if not os.path.exists(path): return None
    d = pd.read_csv(path)
    for k, v in key.items():
        if k not in d.columns: return None
        d = d[d[k] == v]
    return d.iloc[0].to_dict() if len(d) else None

def append_row(path, row):
    d = pd.read_csv(path) if os.path.exists(path) else pd.DataFrame()
    _atomic_csv(pd.concat([d, pd.DataFrame([row])], ignore_index=True), path)

In [ ]:
# ---- rebuild stored models from their saved configuration (no search is repeated) -------------
def params_of(row):
    p = json.loads(row["params_fitted"])
    return {k: v for k, v in p.items()}

def regen_binary(row, variant, source):
    """Refit the stored configuration on the stored split; check the confusion counts match exactly."""
    task, seed = row["task"], int(row["seed"])
    name = "%s__%s_%s_s%d_%s" % (source, variant, task, seed, row.get("strategy", "HGS"))
    cached = load_pred(name)
    if cached is not None:
        return name, cached
    idx, y, tr, te = split_task(task, seed)
    if variant == "no_full_name":
        if SEC_NONAME is None: return name, None
        Xtr, Xte = features("V0", idx, tr, te, sec=SEC_NONAME)
    else:
        Xtr, Xte = features({"main": "V0", "no_tag_correction": "V1"}.get(variant, variant), idx, tr, te)
    m = build_estimator(params_of(row)).fit(Xtr, y[tr]); proba = m.predict_proba(Xte)[:, 1]
    pred = (proba >= 0.5).astype(int)
    tn, fp, fn, tp = confusion_matrix(y[te], pred, labels=[0, 1]).ravel()
    match = (tp, fp, fn, tn) == (int(row["TP"]), int(row["FP"]), int(row["FN"]), int(row["TN"]))
    out = dict(y=y[te], proba=proba, pred=pred, test_rows=idx[te], match=np.array(match))
    save_pred(name, **out)
    return name, out

def regen_4class(row):
    """Same for the four-class runs of notebook 08."""
    variant, split, seed = row["variant"], row["split"], int(row["seed"])
    name = "peerj__%s_%s_s%d_%s" % (variant, split, seed, row.get("strategy", "HGS"))
    cached = load_pred(name)
    if cached is not None:
        return name, cached
    y4 = np.array([{0: 0, 1: 1, 10: 2, 11: 3}[c] for c in RAW_CODE]); idx = np.arange(len(y4))
    tr, te = train_test_split(idx, test_size=0.20, random_state=seed, stratify=y4 if split == "stratified" else None)
    Ztr, Zte = embed_variant(variant, SEC_NAME[tr], SEC_NAME[te])
    Xtr = np.hstack([Ztr, NUM_X[tr]]); Xte = np.hstack([Zte, NUM_X[te]])
    sc = StandardScaler().fit(Xtr); Xtr, Xte = sc.transform(Xtr), sc.transform(Xte)
    est = build_estimator(params_of(row))
    if getattr(est, "solver", "") == "liblinear":
        from sklearn.multiclass import OneVsRestClassifier
        est = OneVsRestClassifier(est)
    est.fit(Xtr, y4[tr]); proba = est.predict_proba(Xte); pred = proba.argmax(1)
    pred = est.classes_[pred] if hasattr(est, "classes_") else pred
    cm = confusion_matrix(y4[te], pred, labels=[0, 1, 2, 3])
    match = cm.tolist() == json.loads(row["confusion"])
    out = dict(y=y4[te], proba=proba, pred=pred, test_rows=te, match=np.array(match))
    save_pred(name, **out)
    return name, out

## Configuration

In [ ]:
B_BOOT = 5000                     # bootstrap resamples per interval
USE_ONLY_EXACT = True             # analyse only rebuilt runs whose confusion counts equal the stored ones
ALL_OUT, BEST_OUT, PEERJ_OUT = [os.path.join(OUT_DIR, f) for f in
                                ("results_all.csv", "results_bestemb.csv", "results_peerj.csv")]
REPORT = []
def say(*a):
    s = " ".join(str(x) for x in a); print(s); REPORT.append(s)

## 1. Rebuild stored models

In [ ]:
R = pd.read_csv(ALL_OUT); BE = pd.read_csv(BEST_OUT) if os.path.exists(BEST_OUT) else pd.DataFrame()
P4 = pd.read_csv(PEERJ_OUT) if os.path.exists(PEERJ_OUT) else pd.DataFrame()
PRED = {}                          # (source, variant, task, seed, strategy) -> predictions
say("=== 1. Rebuilding stored models and checking they reproduce the stored results ===")
for _, row in R.iterrows():
    name, out = regen_binary(row, row["variant"], "main")
    if out is not None: PRED[("main", row["variant"], row["task"], int(row["seed"]), row["strategy"])] = out
for _, row in BE.iterrows():
    name, out = regen_binary(row, row["variant"], "bestemb")
    if out is not None: PRED[("bestemb", row["variant"], row["task"], int(row["seed"]), row["strategy"])] = out
for _, row in P4.iterrows():
    name, out = regen_4class(row)
    PRED[("peerj", row["variant"], row["split"], int(row["seed"]), row["strategy"])] = out
ok = {k: bool(v["match"]) for k, v in PRED.items()}
say("runs rebuilt: %d | exact match with stored confusion counts: %d | differ: %d" % (
    len(ok), sum(ok.values()), len(ok) - sum(ok.values())))
for k, v in ok.items():
    if not v: say("   DIFFERS%s:" % (" (excluded from the tests below)" if USE_ONLY_EXACT else ""), k)
if USE_ONLY_EXACT:
    PRED = {k: v for k, v in PRED.items() if ok[k]}
else:
    say("NOTE: USE_ONLY_EXACT = False - runs that differ from the stored counts are analysed as rebuilt")
if not PRED:
    raise RuntimeError("No stored run was reproduced exactly. Check that OUT_DIR holds the embedding cache used for "
                       "the paper (roberta_sections_name1.npy) and compare library versions with the paper's runs. "
                       "To analyse the rebuilt models anyway, set USE_ONLY_EXACT = False in the configuration cell.")

## 2. Confidence intervals

In [ ]:
say("\n=== 2. 95% bootstrap intervals for the headline results (HGS, stratified bootstrap of the test set) ===")
def auc_metric(y, s): return roc_auc_score(y, s)
rows = []
for task in ["T1", "T1b", "T2", "T3"]:
    for src, var in [("main", "main"), ("bestemb", "V5")]:
        runs = {s: PRED.get((src, var, task, s, "HGS")) for s in (42, 7, 123)}
        runs = {s: r for s, r in runs.items() if r is not None}
        if not runs: continue
        sims = {"f1_pos": [], "accuracy": [], "roc_auc": []}
        for s, r in runs.items():
            for met, fn, key in [("f1_pos", f1_pos_fast, "pred"), ("accuracy", acc_fast, "pred")]:
                v, ci, se, draws = bootstrap_ci(r["y"], r[key], fn, B=B_BOOT, seed=s); sims[met].append(draws)
                rows.append(dict(task=task, variant=var, seed=s, metric=met, value=v, lo=ci[0], hi=ci[1], se=se))
            v, ci, se, draws = bootstrap_ci(r["y"], r["proba"], auc_metric, B=1000, seed=s); sims["roc_auc"].append(draws)
            rows.append(dict(task=task, variant=var, seed=s, metric="roc_auc", value=v, lo=ci[0], hi=ci[1], se=se))
        for met, draws in sims.items():
            m = np.mean([np.mean(d) for d in draws]); pooled = np.mean(draws, 0)   # splits treated as independent
            lo, hi = np.percentile(pooled, [2.5, 97.5])
            rows.append(dict(task=task, variant=var, seed="mean", metric=met, value=m, lo=lo, hi=hi, se=pooled.std(ddof=1)))
            say("%-4s %-4s %-9s mean of %d splits %.4f  95%% CI [%.4f, %.4f]" % (task, "STE" if var == "main" else var, met, len(draws), m, lo, hi))
CI_TABLE = pd.DataFrame(rows, columns=["task", "variant", "seed", "metric", "value", "lo", "hi", "se"])

## 3. Paired tests

In [ ]:
say("\n=== 3. Paired comparisons on identical test profiles ===")
def paired_block(title, key_a, key_b, tasks):
    say("--- %s ---" % title)
    for task in tasks:
        diffs, n_tr, n_te = [], None, None
        for s in (42, 7, 123):
            a, b = PRED.get(key_a(task, s)), PRED.get(key_b(task, s))
            if a is None or b is None: continue
            assert np.array_equal(a["test_rows"], b["test_rows"])
            d, ci, se = paired_bootstrap(a["y"], a["pred"], b["pred"], f1_pos_fast, B=B_BOOT, seed=s)
            _, _, _, p_auc = delong_test(a["y"], a["proba"], b["proba"]); n12, n21, p_mc = mcnemar_exact(a["y"], a["pred"], b["pred"])
            diffs.append(d); n_te = len(a["y"]); n_tr = int(round(n_te * 0.7 / 0.3))
            say("%-4s seed %-4d dF1 %+.4f [%+.4f, %+.4f] | DeLong p %s | McNemar %d/%d p %s | detectable at 80%% power: %.4f" % (
                task, s, d, ci[0], ci[1], fmt_p(p_auc), n12, n21, fmt_p(p_mc), min_detectable_difference(se)))
        if len(diffs) >= 2:
            m, t, p = corrected_resampled_t(diffs, n_tr, n_te)
            say("%-4s across %d seeds: mean dF1 %+.4f | corrected resampled t two-sided p %s" % (task, len(diffs), m, fmt_p(p)))
paired_block("projection (V5) vs STE (V0), both HGS",
             lambda t, s: ("bestemb", "V5", t, s, "HGS"), lambda t, s: ("main", "main", t, s, "HGS"), ["T1", "T1b", "T2", "T3"])
paired_block("HGS vs matched-budget random search (STE)",
             lambda t, s: ("main", "main", t, s, "HGS"), lambda t, s: ("main", "main", t, s, "Random"), ["T1", "T1b", "T2", "T3"])

## 4. Published figures

In [ ]:
say("\n=== 4. Comparison with the published figures ===")
# value, test-set size and positives of the PUBLISHED evaluation (for its own sampling error)
PUBLISHED = [
    dict(task="T1",  variant="main", metric="f1_pos",   value=0.9708, n=720,  n_pos=180, src="Gulati et al., best configuration"),
    dict(task="T1",  variant="main", metric="f1_pos",   value=0.9639, n=720,  n_pos=180, src="Gulati et al., weakest configuration"),
    dict(task="T1b", variant="main", metric="accuracy", value=0.9633, n=360,  n_pos=180, src="Ayoobi et al., best embedding"),
    dict(task="T1b", variant="main", metric="accuracy", value=0.9417, n=360,  n_pos=180, src="Ayoobi et al., weakest embedding"),
    dict(task="T3",  variant="main", metric="f1_pos",   value=0.982,  n=1080, n_pos=540, src="Gulati et al., best retrained"),
    dict(task="T3",  variant="V5",   metric="f1_pos",   value=0.982,  n=1080, n_pos=540, src="Gulati et al., best retrained"),
]
def published_se(metric, value, n, n_pos):
    # binomial standard error; for F1 the positives are used as the effective sample (approximation)
    return np.sqrt(value * (1 - value) / (n if metric == "accuracy" else n_pos))
claims = []
for c in PUBLISHED:
    r = CI_TABLE[(CI_TABLE.task == c["task"]) & (CI_TABLE.variant == c["variant"]) &
                 (CI_TABLE.seed == "mean") & (CI_TABLE.metric == c["metric"])]
    if not len(r): continue
    r = r.iloc[0]; se_pub = published_se(c["metric"], c["value"], c["n"], c["n_pos"])
    z = (r.value - c["value"]) / np.hypot(r.se, se_pub); p2 = 2 * sst.norm.sf(abs(z))
    fixed = "higher" if c["value"] < r.lo else ("lower" if c["value"] > r.hi else "level")
    two = "higher" if (p2 < 0.05 and z > 0) else ("lower" if (p2 < 0.05 and z < 0) else "level")
    claims.append(dict(task=c["task"], variant=c["variant"], metric=c["metric"], ours=r.value, lo=r.lo, hi=r.hi,
                       published=c["value"], source=c["src"], verdict_fixed=fixed, z_two_sample=z, p_two_sample=p2, verdict_two_sample=two))
    say("%-4s %-4s %-8s ours %.4f [%.4f, %.4f] vs %.4f (%s): published value fixed -> %s | two-sample z %.2f p %s -> %s" % (
        c["task"], "STE" if c["variant"] == "main" else c["variant"], c["metric"], r.value, r.lo, r.hi, c["value"],
        c["src"], fixed, z, fmt_p(p2), two))

# four-class task on the partition with the published class counts
say("--- four-class task (Mohiuddin and Almogren) ---")
PJ = dict(macro_f1=0.9608, accuracy=0.9611, macro_recall=0.95)
def macro_rec(y, p):
    return float(np.mean([np.mean(p[y == c] == c) for c in np.unique(y)]))
for var in ["V0", "V5"]:
    r = PRED.get(("peerj", var, "random", 42, "HGS"))
    if r is None: continue
    for met, fn in [("macro_f1", lambda y, p: macro_f1_fast(y, p, 4)), ("accuracy", acc_fast), ("macro_recall", macro_rec)]:
        v, ci, se, _ = bootstrap_ci(r["y"], r["pred"], fn, B=B_BOOT, seed=1)
        se_pub = np.sqrt(PJ[met] * (1 - PJ[met]) / 720); z = (v - PJ[met]) / np.hypot(se, se_pub); p2 = 2 * sst.norm.sf(abs(z))
        fixed = "higher" if PJ[met] < ci[0] else ("lower" if PJ[met] > ci[1] else "level")
        claims.append(dict(task="T4", variant=var, metric=met, ours=v, lo=ci[0], hi=ci[1], published=PJ[met],
                           source="Mohiuddin and Almogren", verdict_fixed=fixed, z_two_sample=z, p_two_sample=p2,
                           verdict_two_sample="higher" if (p2 < 0.05 and z > 0) else ("lower" if (p2 < 0.05 and z < 0) else "level")))
        say("T4   %-4s %-12s ours %.4f [%.4f, %.4f] vs %.4f: published value fixed -> %s | two-sample p %s" % (
            "STE" if var == "V0" else var, met, v, ci[0], ci[1], PJ[met], fixed, fmt_p(p2)))
# partition effect: stratified splits vs the partition with the published class counts
strat = [PRED.get(("peerj", "V0", "stratified", s, "HGS")) for s in (42, 7, 123)]
unstr = PRED.get(("peerj", "V0", "random", 42, "HGS"))
if unstr is not None and all(x is not None for x in strat):
    rng = np.random.default_rng(3)
    def draw(r): i = _strat_resample(r["y"], rng); return macro_f1_fast(r["y"][i], r["pred"][i], 4)
    d = np.array([np.mean([draw(r) for r in strat]) - draw(unstr) for _ in range(2000)])
    obs = np.mean([macro_f1_fast(r["y"], r["pred"], 4) for r in strat]) - macro_f1_fast(unstr["y"], unstr["pred"], 4)
    say("partition effect on macro F1 (stratified mean - published counts): %+.4f, 95%% CI [%+.4f, %+.4f]" % (obs, *np.percentile(d, [2.5, 97.5])))
pd.DataFrame(claims).to_csv(os.path.join(OUT_DIR, "stats_claims.csv"), index=False)

## 5. Summary

In [ ]:
say("\n=== 5. What the paper can claim ===")
CL = pd.DataFrame(claims)
if not len(CL): say("no comparison could be made (no reproduced run for the tasks with published figures)")
for _, c in CL.iterrows():
    word = {"higher": "significantly higher than", "lower": "significantly lower than", "level": "statistically level with"}[c.verdict_two_sample]
    say("- %s %s %s: %.4f is %s %.4f (%s)" % (c.task, "STE" if c.variant in ("main", "V0") else c.variant,
        c.metric, c.ours, word, c.published, c.source))
say("Rule used: 'higher' only when the two-sample test (which counts the sampling error of both evaluations) gives p < 0.05.")
open(os.path.join(OUT_DIR, "stats_report.txt"), "w").write("\n".join(REPORT))
CI_TABLE.to_csv(os.path.join(OUT_DIR, "stats_intervals.csv"), index=False)
print("\nwritten: stats_report.txt, stats_intervals.csv, stats_claims.csv")